# 04 - Graph Model

Builds a fraud relationship graph from shared claim attributes, trains a GCN on all rows, and writes full-row `graph_embeddings.csv` for fusion.

In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score

DATA_PATH = "../data/preprocessed.csv"
OUT_PATH = "../data/graph_embeddings.csv"

df = pd.read_csv(DATA_PATH).sort_values("claim_id").reset_index(drop=True)
target_col = "fraudfound_p"
feature_cols = [c for c in [
    "avg_speed_kmph", "max_speed_kmph", "hard_brakes_per_trip", "rapid_acceleration_events",
    "trip_duration_minutes", "distance_km", "night_driving_ratio", "urban_driving_ratio",
    "harsh_cornering_events", "idle_time_minutes", "speeding_risk", "speed_volatility",
    "harsh_braking_risk", "harsh_acceleration_risk", "harsh_cornering_risk",
    "harsh_driving_index", "high_night_driving", "high_urban_driving", "fast_claim",
    "high_claim_history", "claim_driving_risk", "deductible", "driverrating",
    "policytype", "vehiclecategory", "vehicleprice", "age", "sex", "maritalstatus",
    "ageofvehicle", "accidentarea", "fault", "basepolicy"
] if c in df.columns]

X_np = df[feature_cols].astype(float).to_numpy()
y_np = df[target_col].astype(int).to_numpy()
print("Rows:", len(df), "Graph features:", len(feature_cols))

/home/ahan/Workspace/projects/vehicle_fraud_detection/fraud_env/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Rows: 15420 Graph features: 33


In [2]:
# Build graph edges from claim relationships.
# Nodes are claims. Edges connect claims sharing important insurance/fraud attributes.
relationship_cols = [c for c in [
    "make", "accidentarea", "fault", "policytype", "vehiclecategory",
    "basepolicy", "ageofvehicle", "pastnumberofclaims", "high_claim_history",
] if c in df.columns]

max_edges_per_group = 8
edges = set()

for col in relationship_cols:
    for _, group in df.groupby(col, sort=False):
        nodes = group.index.to_numpy()
        if len(nodes) < 2:
            continue
        # Link each claim to a bounded number of claims with the same relationship value.
        # This keeps the graph meaningful without creating a huge fully connected component.
        for pos, src in enumerate(nodes):
            upper = min(pos + 1 + max_edges_per_group, len(nodes))
            for dst in nodes[pos + 1:upper]:
                edges.add((int(src), int(dst)))
                edges.add((int(dst), int(src)))

if not edges:
    raise ValueError("No graph edges were created. Check relationship_cols.")

edge_index = torch.tensor(list(edges), dtype=torch.long).t().contiguous()
print("Relationship columns:", relationship_cols)
print("Total graph edges:", edge_index.shape[1])

data = Data(
    x=torch.tensor(X_np, dtype=torch.float32),
    edge_index=edge_index,
    y=torch.tensor(y_np, dtype=torch.long),
)
print(data)

Relationship columns: ['make', 'accidentarea', 'fault', 'policytype', 'vehiclecategory', 'basepolicy', 'ageofvehicle', 'pastnumberofclaims', 'high_claim_history']
Total graph edges: 906604
Data(x=[15420, 33], edge_index=[2, 906604], y=[15420])


In [3]:
class FraudGCN(torch.nn.Module):
    def __init__(self, input_dim, hidden_dim=64, embedding_dim=32):
        super().__init__()
        self.conv1 = GCNConv(input_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, embedding_dim)
        self.classifier = GCNConv(embedding_dim, 2)
        self.dropout = torch.nn.Dropout(0.3)

    def forward(self, data):
        x = F.relu(self.conv1(data.x, data.edge_index))
        x = self.dropout(x)
        emb = F.relu(self.conv2(x, data.edge_index))
        logits = self.classifier(emb, data.edge_index)
        return logits, emb

train_idx, test_idx = train_test_split(
    np.arange(len(df)), test_size=0.2, stratify=y_np, random_state=42
)
train_idx = torch.tensor(train_idx, dtype=torch.long)
test_idx = torch.tensor(test_idx, dtype=torch.long)

model = FraudGCN(input_dim=data.x.shape[1])
class_counts = np.bincount(y_np, minlength=2)
weights = torch.tensor([1.0, class_counts[0] / max(class_counts[1], 1)], dtype=torch.float32)
criterion = torch.nn.CrossEntropyLoss(weight=weights)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=5e-4)

for epoch in range(80):
    model.train()
    optimizer.zero_grad()
    logits, _ = model(data)
    loss = criterion(logits[train_idx], data.y[train_idx])
    loss.backward()
    optimizer.step()
    if epoch % 10 == 0:
        with torch.no_grad():
            probs = torch.softmax(logits[test_idx], dim=1)[:, 1].cpu().numpy()
            auc = roc_auc_score(y_np[test_idx.numpy()], probs)
        print(f"Epoch {epoch:03d} | loss={loss.item():.4f} | auc={auc:.4f}")

Epoch 000 | loss=0.6936 | auc=0.4976
Epoch 010 | loss=0.6917 | auc=0.5331
Epoch 020 | loss=0.6903 | auc=0.5435
Epoch 030 | loss=0.6888 | auc=0.5392
Epoch 040 | loss=0.6871 | auc=0.5369
Epoch 050 | loss=0.6855 | auc=0.5383
Epoch 060 | loss=0.6839 | auc=0.5361
Epoch 070 | loss=0.6825 | auc=0.5347


In [4]:
model.eval()
with torch.no_grad():
    logits, emb = model(data)
    probs = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
    embeddings = emb.cpu().numpy()

test_np = test_idx.numpy()
print("ROC-AUC:", roc_auc_score(y_np[test_np], probs[test_np]))
print("PR-AUC:", average_precision_score(y_np[test_np], probs[test_np]))
print(classification_report(y_np[test_np], (probs[test_np] >= 0.5).astype(int), zero_division=0))

out = pd.DataFrame({"claim_id": df["claim_id"].astype(int)})
for i in range(embeddings.shape[1]):
    out[f"graph_emb_{i:03d}"] = embeddings[:, i]
out["graph_fraud_probability"] = probs
out[target_col] = y_np

assert len(out) == len(df)
assert out["claim_id"].is_unique
out.to_csv(OUT_PATH, index=False)
print("Saved:", OUT_PATH, out.shape)
out.head()

ROC-AUC: 0.5370370025078546
PR-AUC: 0.0658565534950527
              precision    recall  f1-score   support

           0       0.95      0.55      0.69      2899
           1       0.07      0.54      0.13       185

    accuracy                           0.55      3084
   macro avg       0.51      0.54      0.41      3084
weighted avg       0.90      0.55      0.66      3084

Saved: ../data/graph_embeddings.csv (15420, 35)


,claim_id,graph_emb_000,graph_emb_001,graph_emb_002,graph_emb_003,graph_emb_004,graph_emb_005,graph_emb_006,graph_emb_007,graph_emb_008,...,graph_emb_024,graph_emb_025,graph_emb_026,graph_emb_027,graph_emb_028,graph_emb_029,graph_emb_030,graph_emb_031,graph_fraud_probability,fraudfound_p
0,1,0.161539,0.008391,0.052489,0.047438,0.188099,0.0,0.0,0.106509,0.0,...,0.0,0.0,0.108763,0.076876,0.139259,0.042459,0.214958,0.245582,0.482736,0
1,2,0.154808,0.009947,0.055453,0.056597,0.226763,0.0,0.0,0.115911,0.0,...,0.0,0.0,0.104370,0.075722,0.142167,0.063763,0.244063,0.249141,0.470694,0
2,3,0.167101,0.016017,0.061690,0.060673,0.226049,0.0,0.0,0.114448,0.0,...,0.0,0.0,0.118112,0.075815,0.134387,0.054765,0.238633,0.264387,0.473058,0
3,4,0.165071,0.000000,0.048880,0.067172,0.228720,0.0,0.0,0.111784,0.0,...,0.0,0.0,0.111857,0.089329,0.142971,0.072743,0.234895,0.262924,0.473997,0
4,5,0.161317,0.001005,0.053247,0.056951,0.216575,0.0,0.0,0.111097,0.0,...,0.0,0.0,0.099899,0.079496,0.138779,0.055064,0.233951,0.247448,0.472597,0
